In [1]:
import os
import glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, Input, mixed_precision
from sklearn.model_selection import train_test_split
import cv2
import keras
print("library loaded successfully")
print("*"*100)

I0000 00:00:1788521165.728635  304049 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


library loaded successfully
****************************************************************************************************


In [2]:
DATA_PATH = '/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2'
POSITIVE_DIR = os.path.join(DATA_PATH, 'imagesAll_positive') # For Polyp classification
NEGATIVE_DIR = os.path.join(DATA_PATH, 'sequenceData/negativeOnly')  # For non polyp classification
MODEL_SAVE_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/lr-ynet.keras"

In [3]:
def process_path_classification(img_path, label):
    """
    Preprocesses input tensors for Classification Dataset.
    Applies identical CLAHE and Sharpening to match the segmentation pipeline.
    """
    def _read_and_process_clf(img_p):
        img_p_str = img_p.numpy().decode('utf-8') if hasattr(img_p, 'numpy') else img_p.decode('utf-8')
        img = cv2.imread(img_p_str)
        if img is None:
            raise FileNotFoundError(f"❌ Image not found at path: {img_p_str}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        
        # ─── STEP 1: CONTRAST ENHANCEMENT (CLAHE) ───
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(lab)
        
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        cl = clahe.apply(l_channel)
        
        limg = cv2.merge((cl, a_channel, b_channel))
        enhanced_img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
        median_blur = cv2.medianBlur(enhanced_img, 3) 
        
        # ─── STEP 2: MILD SHARPENING ───
        sharpened = cv2.addWeighted(enhanced_img, 1.5, median_blur, -0.5, 0)
        
        # ─── STEP 3: NORMALIZATION ───
        normalized_img = sharpened / 255.0
        return normalized_img.astype(np.float32)
    [img_tensor] = tf.py_function(
        _read_and_process_clf, 
        [img_path], 
        [tf.float32]
    )

    img_tensor.set_shape([IMG_SIZE, IMG_SIZE, 3])
    
    return img_tensor, label



def load_classification_paths():
    """Compiles absolute paths for global positive and artifact-heavy negative sequences."""
    valid_exts = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.PNG', '*.JPG', '*.JPEG')
    pos_files, neg_files = [], []
    
    # 1. Gather all positive paths
    for ext in valid_exts:
        pos_files.extend(glob.glob(os.path.join(POSITIVE_DIR, "**", ext), recursive=True))
        
    # 2. Gather all negative paths from seq1 to seq13
    for i in range(1, 14):
        target_folder = os.path.join(NEGATIVE_DIR, f"seq{i}_neg")
        if os.path.exists(target_folder):
            for ext in valid_exts:
                neg_files.extend(glob.glob(os.path.join(target_folder, "**", ext), recursive=True))
                
    # Filter to ensure only valid files are kept
    pos_files = [f for f in pos_files if os.path.isfile(f)]
    neg_files = [f for f in neg_files if os.path.isfile(f)]
    
    # ─── 🚀 🆕 STEP 3: STRICT 50% - 50% BALANCING LOGIC ───
    min_samples = min(len(pos_files), len(neg_files))
    print(f"\n📊 Raw Counts -> Polyp: {len(pos_files)} | Non-Polyp: {len(neg_files)}")
    print(f"⚖️ Balancing Dataset at Sweet Spot: {min_samples} samples each (Total: {min_samples * 2})")
    
    # Randomly shuffle before trimming to ensure diversity
    np.random.seed(42)
    np.random.shuffle(pos_files)
    np.random.shuffle(neg_files)
    
    # Slice both lists to have the exact same size
    pos_files = pos_files[:min_samples]
    neg_files = neg_files[:min_samples]
    print("positive files", len(pos_files), "neg files", len(neg_files))
    # ──────────────────────────────────────────────────────
    
    file_paths = pos_files + neg_files
    labels = [1.0] * len(pos_files) + [0.0] * len(neg_files)
    
    return np.array(file_paths), np.array(labels, dtype=np.float32)

In [4]:
@tf.keras.utils.register_keras_serializable()
class KerasPCCLayer(layers.Layer):
    """Natively computes spatial Pearson Correlation Coefficients within the network graph."""
    def __init__(self, **kwargs):
        super(KerasPCCLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # We used epsilon sothat standard deviation value cannot be zero, otherwise code will crash
        epsilon = 1e-8
        # finds the average standard deviaiton
        mean = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        std = tf.math.reduce_std(inputs, axis=-1, keepdims=True) + epsilon
        # data normalization or (Z-Score)
        norm_inputs = (inputs - mean) / std
        pcc_matrix = tf.matmul(tf.expand_dims(norm_inputs, -1), tf.expand_dims(norm_inputs, 1))
        return layers.Flatten()(pcc_matrix)

In [5]:
@keras.saving.register_keras_serializable()
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)
@keras.saving.register_keras_serializable()
def iou_metric(y_true, y_pred):
    y_true_f = tf.cast(y_true, tf.float32)
    y_pred_f = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + 1.0) / (union + 1.0)
@keras.saving.register_keras_serializable()
def iou_metric(y_true, y_pred):
    y_true_f = tf.cast(y_true, tf.float32)
    y_pred_f = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + 1.0) / (union + 1.0)
@keras.saving.register_keras_serializable()
def iou_metric(y_true, y_pred):
    y_true_f = tf.cast(y_true, tf.float32)
    y_pred_f = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + 1.0) / (union + 1.0)
def bce_dice_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)
def focal_tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, gamma=0.75):
    """
    Focal Tversky Loss for imbalanced medical image segmentation.
    alpha: controls penalty for False Negatives (FN) -> keeping it higher (0.7) focuses on small polyps.
    beta: controls penalty for False Positives (FP).
    """
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    
    # Calculate True Positives, False Negatives, and False Positives
    tp = tf.reduce_sum(y_true_f * y_pred_f)
    fn = tf.reduce_sum(y_true_f * (1.0 - y_pred_f))
    fp = tf.reduce_sum((1.0 - y_true_f) * y_pred_f)
    
    # Tversky Index
    smooth = 1e-6
    tversky_index = (tp + smooth) / (tp + alpha * fn + beta * fp + smooth)
    
    # Focal Tversky Loss
    return tf.math.pow((1.0 - tversky_index), gamma)

In [6]:
import os
import gc
import cv2
import numpy as np
import tensorflow as tf
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

print("\n======================================================")
print(" [ABLATION] OUTER-PRODUCT FEATURES — ALL CLASSIFIERS")
print("=======================================================")
# ─────────────────────────────────────────────────────────────────────────────
# 🛠️ STAGE 1: LOAD PATHS AND DEFINE DATA SPLIT (FIXED VARIABLES)
# ─────────────────────────────────────────────────────────────────────────────
clf_imgs, clf_labels = load_classification_paths()
indices = np.arange(len(clf_imgs))
np.random.seed(42)
np.random.shuffle(indices)
clf_imgs, clf_labels = clf_imgs[indices], clf_labels[indices]

c_split = int(0.8 * len(clf_imgs))

train_imgs_paths = clf_imgs[:c_split]
train_lbls_raw = clf_labels[:c_split]
val_imgs_paths = clf_imgs[c_split:]
val_lbls_raw = clf_labels[c_split:]

IMG_SIZE = 256

MODEL_SAVE_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/thesis_v3_sequential_v_eval_data_driven_v2.keras"

# ─────────────────────────────────────────────────────────────────────────────
# 🧠 STAGE 2: LOAD MODEL & TARGET THE RAW GAP LAYER (NOT THE PCC LAYER)
# ─────────────────────────────────────────────────────────────────────────────
print(f"🔄 Fetching Weights Safely: {MODEL_SAVE_PATH}")
base_model = tf.keras.models.load_model(MODEL_SAVE_PATH, custom_objects={
    'KerasPCCLayer': KerasPCCLayer, 'dice_coef': dice_coef,
    'iou_metric': iou_metric, 'bce_dice_loss': bce_dice_loss,
    'focal_tversky_loss': focal_tversky_loss
}, compile=False)

# 🎯 IMPORTANT: this ablation row extracts the raw 32-dim GAP vector, THEN
# manually computes its self outer-product (32x32 -> flattened 1024-dim),
# WITHOUT the self-normalization step your KerasPCCLayer applies. This isolates
# whether the outer-product expansion itself carries information, separate from
# the normalization your actual PCC layer also does.
target_layer_name = 'clf_gap'
print(f"🎯 Targeting Raw GAP Layer: '{target_layer_name}' — outer product computed manually downstream.")

feature_extractor = tf.keras.Model(
    inputs=base_model.input,
    outputs=base_model.get_layer(target_layer_name).output
)

# ─────────────────────────────────────────────────────────────────────────────
# 🏃‍♂️ STAGE 3: RAW PYTHON FEATURE HARVESTER (With CLAHE & Sharpening Alignment)
# ─────────────────────────────────────────────────────────────────────────────
def pure_python_feature_extractor(img_paths, labels_raw):
    extracted_features = []
    final_labels = []

    total_samples = len(img_paths)
    print(f"⏳ Harvesting outer-product embeddings from {total_samples} samples...")

    for idx in range(total_samples):
        try:
            img = cv2.imread(img_paths[idx])
            if img is None:
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_res = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))

            lab = cv2.cvtColor(img_res, cv2.COLOR_RGB2LAB)
            l_channel, a_channel, b_channel = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            cl = clahe.apply(l_channel)
            limg = cv2.merge((cl, a_channel, b_channel))
            enhanced_img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

            median_blur = cv2.medianBlur(enhanced_img, 3)
            sharpened = cv2.addWeighted(enhanced_img, 1.5, median_blur, -0.5, 0)
            inp = np.expand_dims(sharpened / 255.0, axis=0).astype(np.float32)

            feats = feature_extractor(inp, training=False)
            # feats shape here is (1, 32) directly from GlobalAveragePooling
            gap_vector = tf.reshape(feats, [tf.shape(feats)[0], -1]).numpy().flatten().astype(np.float32)  # (32,)

            # ── Outer product (NO self-normalization — that's what makes this
            # different from your KerasPCCLayer) ──────────────────────────────
            outer = np.outer(gap_vector, gap_vector)          # (32, 32)
            feats_flat = outer.flatten().astype(np.float32)   # (1024,)

            lbl = labels_raw[idx]
            if isinstance(lbl, (list, tuple, np.ndarray)):
                binary_label = lbl[-1] if len(lbl) > 1 else lbl[0]
            else:
                binary_label = lbl

            if hasattr(binary_label, 'numpy'):
                binary_label = binary_label.numpy()
            if isinstance(binary_label, (np.ndarray, list)) and len(binary_label) == 1:
                binary_label = binary_label[0]

            extracted_features.append(feats_flat)
            final_labels.append(int(binary_label))

        except Exception as e:
            continue

        if idx % 100 == 0:
            gc.collect()

    return np.array(extracted_features, dtype=np.float32), np.array(final_labels, dtype=np.int32)

print("\n🎬 Extracting Training Features...")
X_train_raw, y_train = pure_python_feature_extractor(train_imgs_paths, train_lbls_raw)

print("\n🎬 Extracting Validation Features...")
X_val_raw, y_val = pure_python_feature_extractor(val_imgs_paths, val_lbls_raw)

y_train = y_train.ravel()
y_val = y_val.ravel()

print(f"\n✅ Outer-Product Feature Extraction Complete! Feature dim: {X_train_raw.shape[1]}")
print(f" -> Train Labels Distribution (0: Normal / 1: Polyp): {np.bincount(y_train)}")
print(f" -> Val Labels Distribution   (0: Normal / 1: Polyp): {np.bincount(y_val)}")

del base_model
del feature_extractor
tf.keras.backend.clear_session()
gc.collect()

# ─────────────────────────────────────────────────────────────────────────────
# 📉 STAGE 4: NO SELECTION, NO SCALING — RAW OUTER-PRODUCT BASELINE
# ─────────────────────────────────────────────────────────────────────────────
# Deliberately skipped here too, same reasoning as the Raw GAP row: this row
# isolates the outer-product expansion ALONE. If you also ran SelectKBest +
# RobustScaler on this, you'd no longer be able to tell whether a result came
# from the outer product itself or from those extra steps (which your "full
# PCC pipeline" row already covers).
X_train_final = X_train_raw
X_val_final = X_val_raw

np.savez("outer_product_features.npz",
         X_train=X_train_final, y_train=y_train,
         X_val=X_val_final, y_val=y_val)
print("💾 Saved outer-product feature arrays to outer_product_features.npz")

# ─────────────────────────────────────────────────────────────────────────────
# 🎯 SHARED EVALUATION FUNCTION (same G-mean sweep + metrics as all other scripts)
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_probs(final_probs, y_true, set_name, model_name):
    best_thresh = 0.5
    best_gmean = 0
    best_preds = np.zeros_like(y_true)

    for thresh in np.arange(0.05, 0.95, 0.02):
        current_preds = (final_probs > thresh).astype(int)
        tp = np.sum((current_preds == 1) & (y_true == 1))
        fp = np.sum((current_preds == 1) & (y_true == 0))
        fn = np.sum((current_preds == 0) & (y_true == 1))
        tn = np.sum((current_preds == 0) & (y_true == 0))
        sensitivity = tp / (tp + fn + 1e-7)
        specificity = tn / (tn + fp + 1e-7)
        gmean = np.sqrt(sensitivity * specificity)
        if gmean > best_gmean:
            best_gmean = gmean
            best_thresh = thresh
            best_preds = current_preds

    tp = np.sum((best_preds == 1) & (y_true == 1))
    fp = np.sum((best_preds == 1) & (y_true == 0))
    fn = np.sum((best_preds == 0) & (y_true == 1))
    tn = np.sum((best_preds == 0) & (y_true == 0))

    acc = np.mean(best_preds == y_true)
    precision = tp / (tp + fp + 1e-7)
    recall = tp / (tp + fn + 1e-7)
    specificity = tn / (tn + fp + 1e-7)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-7)
    roc_auc = roc_auc_score(y_true, final_probs)
    prec_vals, rec_vals, _ = precision_recall_curve(y_true, final_probs)
    pr_auc = auc(rec_vals, prec_vals)

    print(f"\n🎯 [{model_name} | {set_name} | Threshold={best_thresh:.2f}]:")
    print(f"    Accuracy    : {acc*100:.2f}%")
    print(f"    Precision   : {precision*100:.2f}%")
    print(f"    Recall(Sens): {recall*100:.2f}%")
    print(f"    Specificity : {specificity*100:.2f}%")
    print(f"    F1-Score    : {f1*100:.2f}%")
    print(f"    G-Mean      : {best_gmean*100:.2f}%")
    print(f"    ROC-AUC     : {roc_auc*100:.2f}% 🎯")
    print(f"    PR-AUC      : {pr_auc*100:.2f}% 📈")

    return {"accuracy": acc, "precision": precision, "recall": recall,
            "specificity": specificity, "f1": f1, "g_mean": best_gmean,
            "roc_auc": roc_auc, "pr_auc": pr_auc}

results = {}

# ── Logistic Regression ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🧮 Logistic Regression on Outer-Product Features")
lr = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=2000,
                         class_weight='balanced', random_state=42)
lr.fit(X_train_final, y_train)
results["LogisticRegression"] = {
    "train": evaluate_probs(lr.predict_proba(X_train_final)[:, 1], y_train, "Training", "LR"),
    "val": evaluate_probs(lr.predict_proba(X_val_final)[:, 1], y_val, "Validation", "LR"),
}

# ── Linear SVM ───────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🧮 Linear SVM on Outer-Product Features")
base_svm = LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, random_state=42)
svm = CalibratedClassifierCV(base_svm, method='sigmoid', cv=5)
svm.fit(X_train_final, y_train)
results["LinearSVM"] = {
    "train": evaluate_probs(svm.predict_proba(X_train_final)[:, 1], y_train, "Training", "SVM"),
    "val": evaluate_probs(svm.predict_proba(X_val_final)[:, 1], y_val, "Validation", "SVM"),
}

# ── Ridge Classifier ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🧮 Ridge Classifier on Outer-Product Features")
base_ridge = RidgeClassifier(alpha=1.0, class_weight='balanced', random_state=42)
ridge = CalibratedClassifierCV(base_ridge, method='sigmoid', cv=5)
ridge.fit(X_train_final, y_train)
results["Ridge"] = {
    "train": evaluate_probs(ridge.predict_proba(X_train_final)[:, 1], y_train, "Training", "Ridge"),
    "val": evaluate_probs(ridge.predict_proba(X_val_final)[:, 1], y_val, "Validation", "Ridge"),
}

# ── Single ELM ────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🧮 Single ELM on Outer-Product Features")
np.random.seed(42)
input_dim = X_train_final.shape[1]
hidden_neurons = 1024
W_input = np.random.normal(0, 0.05, (input_dim, hidden_neurons)).astype(np.float32)
b_input = np.random.normal(0, 0.05, (1, hidden_neurons)).astype(np.float32)
H_train = np.tanh(np.dot(X_train_final, W_input) + b_input)
H_val = np.tanh(np.dot(X_val_final, W_input) + b_input)
I = np.eye(hidden_neurons, dtype=np.float32)
reg_lambda = 1e-2
y_train_mapped = np.where(y_train == 1, 1.0, -1.0).astype(np.float32)
W_output = np.linalg.inv(np.dot(H_train.T, H_train) + reg_lambda * I) @ np.dot(H_train.T, y_train_mapped)

def elm_probs(H_matrix):
    raw = np.dot(H_matrix, W_output)
    return (raw - raw.min()) / (raw.max() - raw.min() + 1e-7)

results["SingleELM"] = {
    "train": evaluate_probs(elm_probs(H_train), y_train, "Training", "Single ELM"),
    "val": evaluate_probs(elm_probs(H_val), y_val, "Validation", "Single ELM"),
}

# ── Ensemble ELM ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🧮 Ensemble ELM on Outer-Product Features")
lambdas = [1e-3, 1e-2, 1.0, 10.0]
ensemble_weights = []
for lam in lambdas:
    W_out = np.linalg.inv(np.dot(H_train.T, H_train) + lam * I) @ np.dot(H_train.T, y_train_mapped)
    ensemble_weights.append(W_out)

def ensemble_elm_probs(H_matrix):
    all_raw = [np.dot(H_matrix, W_out) for W_out in ensemble_weights]
    final_scores = np.mean(all_raw, axis=0)
    return (final_scores - final_scores.min()) / (final_scores.max() - final_scores.min() + 1e-7)

results["EnsembleELM"] = {
    "train": evaluate_probs(ensemble_elm_probs(H_train), y_train, "Training", "Ensemble ELM"),
    "val": evaluate_probs(ensemble_elm_probs(H_val), y_val, "Validation", "Ensemble ELM"),
}

# ─────────────────────────────────────────────────────────────────────────────
# 📊 FINAL SUMMARY TABLE (Validation set) — copy straight into your ablation table
# ─────────────────────────────────────────────────────────────────────────────
print("\n\n📊 ===== OUTER-PRODUCT FEATURES — VALIDATION SUMMARY =====")
print(f"{'Model':<18}{'Acc':>8}{'Prec':>8}{'Recall':>8}{'Spec':>8}{'F1':>8}{'GMean':>8}{'ROC-AUC':>10}{'PR-AUC':>9}")
for name, r in results.items():
    v = r["val"]
    print(f"{name:<18}{v['accuracy']*100:>7.2f}%{v['precision']*100:>7.2f}%{v['recall']*100:>7.2f}%"
          f"{v['specificity']*100:>7.2f}%{v['f1']*100:>7.2f}%{v['g_mean']*100:>7.2f}%"
          f"{v['roc_auc']*100:>9.2f}%{v['pr_auc']*100:>8.2f}%")


 [ABLATION] OUTER-PRODUCT FEATURES — ALL CLASSIFIERS

📊 Raw Counts -> Polyp: 3762 | Non-Polyp: 2520
⚖️ Balancing Dataset at Sweet Spot: 2520 samples each (Total: 5040)
positive files 2520 neg files 2520
🔄 Fetching Weights Safely: /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/thesis_v3_sequential_v_eval_data_driven_v2.keras


W0000 00:00:1788521289.659712  304049 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1788521289.660199  304049 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


🎯 Targeting Raw GAP Layer: 'clf_gap' — outer product computed manually downstream.

🎬 Extracting Training Features...
⏳ Harvesting outer-product embeddings from 4032 samples...


I0000 00:00:1788521294.480984  304049 cuda_dnn.cc:461] Loaded cuDNN version 92101



🎬 Extracting Validation Features...
⏳ Harvesting outer-product embeddings from 1008 samples...

✅ Outer-Product Feature Extraction Complete! Feature dim: 1024
 -> Train Labels Distribution (0: Normal / 1: Polyp): [1987 2045]
 -> Val Labels Distribution   (0: Normal / 1: Polyp): [533 475]
💾 Saved outer-product feature arrays to outer_product_features.npz

🧮 Logistic Regression on Outer-Product Features

🎯 [LR | Training | Threshold=0.50]:
    Accuracy    : 49.28%
    Precision   : 0.00%
    Recall(Sens): 0.00%
    Specificity : 100.00%
    F1-Score    : 0.00%
    G-Mean      : 0.00%
    ROC-AUC     : 50.00% 🎯
    PR-AUC      : 75.36% 📈

🎯 [LR | Validation | Threshold=0.50]:
    Accuracy    : 52.88%
    Precision   : 0.00%
    Recall(Sens): 0.00%
    Specificity : 100.00%
    F1-Score    : 0.00%
    G-Mean      : 0.00%
    ROC-AUC     : 50.00% 🎯
    PR-AUC      : 73.56% 📈

🧮 Linear SVM on Outer-Product Features


/mnt/c/development/Thesis/PotatoDiseaseClassification-CNN/linux-venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



🎯 [SVM | Training | Threshold=0.59]:
    Accuracy    : 69.92%
    Precision   : 75.81%
    Recall(Sens): 59.76%
    Specificity : 80.37%
    F1-Score    : 66.83%
    G-Mean      : 69.30%
    ROC-AUC     : 78.79% 🎯
    PR-AUC      : 81.44% 📈

🎯 [SVM | Validation | Threshold=0.59]:
    Accuracy    : 70.93%
    Precision   : 72.30%
    Recall(Sens): 62.11%
    Specificity : 78.80%
    F1-Score    : 66.82%
    G-Mean      : 69.96%
    ROC-AUC     : 77.83% 🎯
    PR-AUC      : 79.24% 📈

🧮 Ridge Classifier on Outer-Product Features

🎯 [Ridge | Training | Threshold=0.59]:
    Accuracy    : 69.92%
    Precision   : 75.84%
    Recall(Sens): 59.71%
    Specificity : 80.42%
    F1-Score    : 66.81%
    G-Mean      : 69.29%
    ROC-AUC     : 78.72% 🎯
    PR-AUC      : 81.36% 📈

🎯 [Ridge | Validation | Threshold=0.61]:
    Accuracy    : 71.03%
    Precision   : 73.52%
    Recall(Sens): 60.21%
    Specificity : 80.68%
    F1-Score    : 66.20%
    G-Mean      : 69.70%
    ROC-AUC     : 77.74% 🎯
    P